# Fine-Tuning Qwen2.5-3B-Instruct with Unsloth in Google Colab

> **Note:** Make sure you are using a GPU runtime in Colab (`Runtime` -> `Change runtime type` -> select `T4 GPU` or better).

## 1. Install Dependencies

In [ ]:
%%capture
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "xformers<0.0.27" "trl<0.9.0" peft accelerate bitsandbytes

## 2. Load Model & Tokenizer
We load `Qwen2.5-3B-Instruct` with 4-bit quantization and configure LoRA adapters.

In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048
dtype = None  # None for auto-detection (Float16 for T4, Bfloat16 for Ampere+)
load_in_4bit = True

# 1. Load Pre-trained Base Model
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen2.5-3B-Instruct-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

# 2. Add LoRA Adapters
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth", # 4x longer context, 0% overhead
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)

## 3. Data Preparation & Formatting
Define prompt styles and format dataset examples.

In [ ]:
from datasets import Dataset

# Training Prompt Template
train_prompt_style = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
Analyze the following education and career details.

### Input:
{}

### Response:
Degree: {}
Roles: {}
Goals: {}
Summary: {}"""

EOS_TOKEN = tokenizer.eos_token

def formatting_prompts_func(examples):
    inputs       = examples["input"]
    degrees      = examples["degree"]
    roles        = examples["roles"]
    goals        = examples["goals"]
    summaries    = examples["summary"]
    texts = []
    for i, d, r, g, s in zip(inputs, degrees, roles, goals, summaries):
        # Must append EOS_TOKEN so the model learns when to stop generating
        text = train_prompt_style.format(i, d, r, g, s) + EOS_TOKEN
        texts.append(text)
    return { "text" : texts }

# Sample Dataset (Replace this with your actual CSV/JSON data)
sample_data = {
    "input": [
        "I have a Bachelor's degree in Computer Science, and I've worked as a Software Engineer for 5 years. My goal is to become a Lead Developer.",
        "Completed B.Tech in Electronics, worked 3 years as Data Analyst. Aiming for Senior Machine Learning Engineer."
    ],
    "degree": ["Bachelor's in Computer Science", "B.Tech in Electronics"],
    "roles": ["Software Engineer (5 years)", "Data Analyst (3 years)"],
    "goals": ["Lead Developer", "Senior Machine Learning Engineer"],
    "summary": [
        "Experienced Software Engineer with a CS background aiming to step into technical leadership as a Lead Developer.",
        "Data Analyst with an Electronics background striving to transition into ML engineering."
    ]
}

# Load dataset (Or use load_dataset("json", data_files="your_data.json") or load_dataset("csv", data_files="your_data.csv"))
dataset = Dataset.from_dict(sample_data)
dataset = dataset.map(formatting_prompts_func, batched = True)

## 4. Fine-Tuning with SFTTrainer

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False, # Set True for short sequences to pack multiple into 1 prompt
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 60,
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
    ),
)

trainer_stats = trainer.train()

## 5. Save the Model & Adapters

In [ ]:
# Save LoRA adapters locally in Colab
model.save_pretrained("lora_model")
tokenizer.save_pretrained("lora_model")

# (Optional) To save merged 16-bit model or export to GGUF:
# model.save_pretrained_merged("model_merged", tokenizer, save_method = "merged_16bit")
# model.save_pretrained_gguf("model_gguf", tokenizer, quantization_method = "q4_k_m")

## 6. Model Inference (Testing Trained Model)

In [ ]:
# 1. Load saved model with LoRA adapter
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "lora_model", # Load directly from saved directory
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

# 2. Enable Fast Inference Mode
FastLanguageModel.for_inference(model)

# 3. Prepare Inference Prompt (Prompt stops at ### Response: so model generates Degree/Roles/Goals/Summary)
inference_prompt_style = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
Analyze the following education and career details.

### Input:
{}

### Response:
"""

input_text = "I have a Bachelor's degree in Computer Science, and I've worked as a Software Engineer for 5 years. My goal is to become a Lead Developer at a tech company."
formatted_input = inference_prompt_style.format(input_text)

# 4. Tokenize & Generate
inputs = tokenizer([formatted_input], return_tensors = "pt").to("cuda")

outputs = model.generate(
    **inputs,
    max_new_tokens = 256,
    use_cache = True,
    temperature = 0.5,
    min_p = 0.1,
)

# 5. Decode & Print Output
output_text = tokenizer.batch_decode(outputs, skip_special_tokens=True)[0]

response_start = output_text.find("### Response:")
if response_start != -1:
    generated_response = output_text[response_start + len("### Response:") :].strip()
    print("--- Generated Output ---")
    print(generated_response)
else:
    print(output_text)